# Order Tracker
Query and visualize order history from the PYR▲MYD2 research agent.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent:
    if (root / "pyproject.toml").exists():
        break
    root = root.parent
sys.path.insert(0, str(root))

import sqlite3
import pandas as pd
from db import DB_PATH

conn = sqlite3.connect(str(DB_PATH))
print(f"Connected to {DB_PATH}")

Connected to /home/mokuaxv/croo/pyrmyd2/orders.db


## All Orders

In [4]:
df = pd.read_sql_query("SELECT * FROM orders ORDER BY created_at DESC", conn)
if df.empty:
    print("No orders yet.")
else:
    display(df[["order_id", "status", "topic", "word_count", "max_analysts", "created_at", "completed_at", "report_length"]])

No orders yet.


## Statistics

In [3]:
stats = pd.read_sql_query("""
    SELECT
        COUNT(*) as total_orders,
        SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) as completed,
        SUM(CASE WHEN status = 'failed' THEN 1 ELSE 0 END) as failed,
        SUM(CASE WHEN status = 'researching' THEN 1 ELSE 0 END) as in_progress,
        SUM(CASE WHEN status = 'received' THEN 1 ELSE 0 END) as pending,
        AVG(report_length) as avg_report_words,
        SUM(report_length) as total_words
    FROM orders
""", conn)

display(stats)

,total_orders,completed,failed,in_progress,pending,avg_report_words,total_words
0,0,None,None,None,None,None,None


## Filter by Status

In [4]:
# Change this to filter: 'received', 'researching', 'completed', 'failed'
STATUS_FILTER = "completed"

filtered = pd.read_sql_query(
    "SELECT * FROM orders WHERE status = ? ORDER BY created_at DESC",
    conn, params=(STATUS_FILTER,)
)
print(f"{len(filtered)} order(s) with status '{STATUS_FILTER}')")
display(filtered)

0 order(s) with status 'completed')


,id,order_id,negotiation_id,status,topic,word_count,max_analysts,requester_agent_id,service_id,price,created_at,paid_at,completed_at,failed_at,error_message,report_length


## Failed Orders (with errors)

In [5]:
failed = pd.read_sql_query(
    "SELECT order_id, topic, error_message, failed_at FROM orders WHERE status = 'failed' ORDER BY failed_at DESC",
    conn
)
if failed.empty:
    print("No failed orders.")
else:
    display(failed)

No failed orders.


## Order Timeline

In [6]:
timeline = pd.read_sql_query("""
    SELECT
        order_id,
        topic,
        status,
        created_at as received,
        paid_at,
        completed_at,
        failed_at
    FROM orders
    ORDER BY created_at DESC
    LIMIT 10
""", conn)
display(timeline)

,order_id,topic,status,received,paid_at,completed_at,failed_at


In [7]:
conn.close()